# Phase 2 — Data Preparation & Feature Engineering

## Objective

Transform the raw last-mile delivery dataset into a clean, validated,
and model-ready dataset while preserving reproducibility.

The preparation pipeline will:

- Standardize missing-value representations
- Normalize string and categorical fields
- Convert numerical variables to appropriate data types
- Parse temporal variables
- Validate and handle temporal anomalies
- Validate geographic coordinates
- Define the machine learning target
- Engineer temporal and geospatial features
- Validate the resulting feature set
- Save the processed dataset for downstream modeling

The raw dataset will remain unchanged.

All transformations performed in this phase should be reproducible and
suitable for integration into the project's production data pipeline.

In [62]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [63]:
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

In [64]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "India-Food-Delivery-Time-Prediction.txt"
)

INTERIM_DATA_PATH = (
    PROJECT_ROOT    
    /"data"
    /"interim"
)

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    /'data'
    /'processed'
)
print("Project root :",PROJECT_ROOT)
print('Raw data :',RAW_DATA_PATH)
print("Raw data exists :",RAW_DATA_PATH.exists())

Project root : c:\Users\A MAHESH\OneDrive\Desktop\INNOMATICS-DS\ML project\Last-mile-Delivery-Intillegence
Raw data : c:\Users\A MAHESH\OneDrive\Desktop\INNOMATICS-DS\ML project\Last-mile-Delivery-Intillegence\data\raw\India-Food-Delivery-Time-Prediction.txt
Raw data exists : True


In [65]:
# LOAD Raw JSON

with open(RAW_DATA_PATH, "r" , encoding='utf-8')as file :
    raw_data = json.load(file)

df_raw = pd.DataFrame(raw_data)

print(f"Rows  : {df_raw.shape[0]:,}")
print(f"columns : {df_raw.shape[1]}")

df_raw


Rows  : 41,953
columns : 20


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:33:33,11:45:29,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:37,19:51:49,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,8:32:58,8:48:47,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:03:58,18:12:52,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:34:16,13:45:36,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41948,0x1178,RANCHIRES16DEL01,35,4.2,23.371292,85.327872,23.481292,85.437872,08-03-2022,21:47:47,21:59:27,conditions Windy,Jam,2,Drinks,motorcycle,1,No,Metropolitian,(min) 33
41949,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:36:22,11:48:07,conditions Windy,High,1,Meal,motorcycle,0,No,Metropolitian,(min) 32
41950,0x4f8d,CHENRES08DEL03,30,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,23:50:56,0:08:55,conditions Cloudy,Low,1,Drinks,scooter,0,No,Metropolitian,(min) 16
41951,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:40:13,13:42:00,conditions Cloudy,High,0,Snack,motorcycle,1,No,Metropolitian,(min) 26


In [66]:
# establising a raw data snapshot

raw_shape = df_raw.shape
raw_columns = df_raw.columns.tolist()

print("Raw shape :",raw_shape)
print("Raw columns :")
for column in raw_columns :
    print(f"- {column}")

Raw shape : (41953, 20)
Raw columns :
- ID
- Delivery_person_ID
- Delivery_person_Age
- Delivery_person_Ratings
- Restaurant_latitude
- Restaurant_longitude
- Delivery_location_latitude
- Delivery_location_longitude
- Order_Date
- Time_Orderd
- Time_Order_picked
- Weatherconditions
- Road_traffic_density
- Vehicle_condition
- Type_of_order
- Type_of_vehicle
- multiple_deliveries
- Festival
- City
- Time_taken(min)


In [67]:
# Sanity Check
assert df_raw.shape == raw_shape
assert df_raw.columns.tolist() == raw_columns

In [68]:
# we will create the working Dataframe

df_clean = df_raw.copy()

In [69]:
# we will convert common strings representations into actual pandas missing values

missing_tokens = [
     "",
    " ",
    "NaN",
    "nan",
    "NA",
    "N/A",
    "null",
    "None"
 ]
for column in df_clean.select_dtypes(include=['object','str']).columns :
    df_clean[column] = df_clean[column].replace(missing_tokens,np.nan)
 

In [70]:
missing_after_standardization = (df_clean.isna().sum().sort_values(ascending = False))
missing_after_standardization[missing_after_standardization > 0]

Time_Orderd    1600
dtype: int64

In [71]:
# Strip Whitespace
string_columns = df_clean.select_dtypes(
    include=["object","str"]
).columns
for column in string_columns :
    df_clean[column] = (
        df_clean[column].astype("string").str.strip()
    )

In [72]:
whitespace_after = {}

for column in string_columns:
    values = df_clean[column].dropna().astype("string")

    whitespace_after[column] = (
        values.str.strip() != values
    ).sum()

pd.Series(
    whitespace_after,
    name="remaining_outer_whitespace"
)

ID                         0
Delivery_person_ID         0
Delivery_person_Age        0
Delivery_person_Ratings    0
Order_Date                 0
Time_Orderd                0
Time_Order_picked          0
Weatherconditions          0
Road_traffic_density       0
Type_of_order              0
Type_of_vehicle            0
multiple_deliveries        0
Festival                   0
City                       0
Time_taken(min)            0
Name: remaining_outer_whitespace, dtype: int64

In [73]:
# Clean weather Prefix
df_clean["Weatherconditions"] = (
    df_clean["Weatherconditions"].str.replace("conditions ", "", regex=False).str.strip()
)


In [74]:
# checking the Weather conditions
df_clean["Weatherconditions"].value_counts(dropna=False)

Weatherconditions
Fog           7012
Stormy        6974
Cloudy        6932
Sandstorms    6906
Windy         6832
Sunny         6728
NaN            569
Name: count, dtype: Int64

In [75]:
# we will convert numerical variables

numeric_columns = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Vehicle_condition",
    "multiple_deliveries",
]
for column in numeric_columns :
    df_clean[column] = pd.to_numeric(df_clean[column],errors = 'coerce')

In [76]:
df_clean[numeric_columns].dtypes

Delivery_person_Age              Int64
Delivery_person_Ratings        Float64
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Vehicle_condition                int64
multiple_deliveries              Int64
dtype: object

In [77]:
# clean the target
df_clean["target_delivery_time"] = (
    df_clean["Time_taken(min)"]
    .astype("string")
    .str.replace("(min)","",regex=False).str.strip()
)
df_clean["target_delivery_time"] = pd.to_numeric(
    df_clean["target_delivery_time"],
    errors = 'coerce'
)

In [78]:
df_clean['target_delivery_time'].describe()

count      41953.0
mean     26.311539
std       9.380753
min           10.0
25%           19.0
50%           26.0
75%           32.0
max           54.0
Name: target_delivery_time, dtype: Float64

In [79]:
# converting the orderDate to actual format which is in strings

df_clean['order_date'] = pd.to_datetime(
    df_clean["Order_Date"],
    format = "%d-%m-%Y",
    errors = "coerce"
)
print("Invalid order dates:",df_clean['order_date'].isna().sum())
print("Minimum:",df_clean["order_date"].min())
print("Maximum:",df_clean['order_date'].max())

Invalid order dates: 0
Minimum: 2022-02-11 00:00:00
Maximum: 2022-04-06 00:00:00


In [80]:
# Preserving original time columns

df_clean['order_time'] = pd.to_timedelta(
    df_clean["Time_Orderd"],
    errors = 'coerce'
)

df_clean['pickup_time'] = pd.to_timedelta(
    df_clean["Time_Order_picked"],
    errors = 'coerce'
)

In [81]:
# Lets check
print("invalid order times :",df_clean['order_time'].isna().sum())

print("Invalid pickup times :",df_clean['pickup_time'].isna().sum())

invalid order times : 1600
Invalid pickup times : 0


In [82]:
# Investigating invalid order-time values

invalid_order_time_mask = df_clean['order_time'].isna()
print("Number of invalid order times :",invalid_order_time_mask.sum())

print(df_clean.loc[
    invalid_order_time_mask,'Time_Orderd'].value_counts(dropna=False))

Number of invalid order times : 1600
Time_Orderd
<NA>    1600
Name: count, dtype: Int64


In [83]:
# Inspect affected records

df_clean.loc[
    invalid_order_time_mask,
    [
        "ID",
        "Order_Date",
        "Time_Orderd",
        "Time_Order_picked",
        "Time_taken(min)"
    ]
].head(30)

,ID,Order_Date,Time_Orderd,Time_Order_picked,Time_taken(min)
41,0x4f0,06-04-2022,<NA>,18:36:55,(min) 33
66,0xa9f,11-03-2022,<NA>,15:31:11,(min) 19
110,0x6e2,02-04-2022,<NA>,18:26:07,(min) 25
129,0x1ec7,09-03-2022,<NA>,23:41:40,(min) 19
162,0x1b58,12-03-2022,<NA>,17:50:59,(min) 43
228,0x1b50,08-03-2022,<NA>,21:28:01,(min) 43
264,0x4db,05-03-2022,<NA>,22:06:09,(min) 12
280,0x6a7,04-04-2022,<NA>,19:47:07,(min) 39
319,0x4c1,02-03-2022,<NA>,0:07:17,(min) 26
324,0xcee,20-03-2022,<NA>,23:08:46,(min) 42


In [84]:
# Checking Geographical columns

latitude_columns = [
    "Restaurant_latitude",
    "Delivery_location_latitude"
]

longitude_columns = [
    "Restaurant_longitude",
    "Delivery_location_longitude"
]

for column in latitude_columns :
    invalid = ~df_clean[column].between(-90,90)
    print(f"{column} : {invalid.sum()} invalid values")

for column in longitude_columns :
    invalid = ~df_clean[column].between(-180,180)
    print(f"{column} : {invalid.sum()} invalid values")

Restaurant_latitude : 0 invalid values
Delivery_location_latitude : 0 invalid values
Restaurant_longitude : 0 invalid values
Delivery_location_longitude : 0 invalid values


In [85]:
# create pickup Timestamp
df_clean['pickup_timestamp'] = (
    df_clean["order_date"] +
    df_clean['pickup_time']
)

In [86]:
# check

print("invalid_Timestamps :",df_clean['pickup_timestamp'].isna().sum())

invalid_Timestamps : 0


In [87]:
# extracting pikcup time features

df_clean['pickup_hour'] = (df_clean['pickup_timestamp'].dt.hour)
df_clean['pickup_minute'] = (df_clean['pickup_timestamp'].dt.minute)
df_clean['pickup_day_of_week'] = (df_clean['pickup_timestamp'].dt.day_of_week)
df_clean['pickup_day_name'] = (df_clean['pickup_timestamp'].dt.day_name())
df_clean['pickup_is_weekend'] = (df_clean['pickup_day_of_week'] >= 5)


In [88]:
df_clean [
    [ 'pickup_timestamp',
     'pickup_hour',
     'pickup_minute',
     'pickup_day_of_week',
     'pickup_day_name',
     'pickup_is_weekend']
].head(10)

,pickup_timestamp,pickup_hour,pickup_minute,pickup_day_of_week,pickup_day_name,pickup_is_weekend
0,2022-03-19 11:45:29,11,45,5,Saturday,True
1,2022-03-25 19:51:49,19,51,4,Friday,False
2,2022-03-19 08:48:47,8,48,5,Saturday,True
3,2022-04-05 18:12:52,18,12,1,Tuesday,False
4,2022-03-26 13:45:36,13,45,5,Saturday,True
5,2022-03-11 21:30:20,21,30,4,Friday,False
6,2022-03-04 19:34:16,19,34,4,Friday,False
7,2022-03-14 17:31:55,17,31,0,Monday,False
8,2022-03-20 21:06:20,21,6,6,Sunday,True
9,2022-02-12 22:10:28,22,10,5,Saturday,True


In [89]:
# creating_time_of_day_categories
def get_time_period(hour) :
    if 5 <= hour < 11 :
        return 'morning'
    if 11<= hour < 15 :
        return 'afternoon'
    if 15 <= hour < 19 :
        return 'evening'
    if 19 <= hour < 24 :
        return 'night'
    else :
        return 'late_night'

In [90]:
df_clean['pickup_time_period'] = (df_clean['pickup_hour'].apply(get_time_period))

In [91]:
# Check
df_clean['pickup_time_period'].value_counts()

pickup_time_period
night         21828
evening        9327
morning        5200
afternoon      4386
late_night     1212
Name: count, dtype: int64

In [92]:
# create cyclic time features
df_clean["pickup_hour_sin"] = np.sin(2*np.pi * df_clean['pickup_hour'] / 24)
df_clean['pickup_hour_cos'] = np.cos(2*np.pi * df_clean['pickup_hour'] / 24)

In [93]:
# order to pickup duration
df_clean["order_timestamp"] = (df_clean['order_date'] + df_clean['order_time']) 

In [94]:
df_clean['order_to_pickup_minutes'] = (
    df_clean['pickup_timestamp'] - df_clean['order_timestamp']).dt.total_seconds() / 60



In [95]:
df_clean["order_to_pickup_minutes"].describe()

count    40353.000000
mean       -16.037530
std        189.788237
min      -1439.500000
25%          5.516667
50%          9.283333
75%         13.133333
max         19.883333
Name: order_to_pickup_minutes, dtype: float64

In [96]:
# investigate negative durations

negative_mask = (
    df_clean["order_to_pickup_minutes"] < 0 )

print (
    'Negative_order_to_pickup_durations :',
    negative_mask.sum()
)


Negative_order_to_pickup_durations : 794


In [97]:
df_clean.loc[
    negative_mask,
    [
        "Order_Date",
        'Time_Orderd',
        "Time_Order_picked",
        'order_to_pickup_minutes'
    ]
].head(20)

,Order_Date,Time_Orderd,Time_Order_picked,order_to_pickup_minutes
32,11-02-2022,13:40:46,13:40:35,-0.183333
67,05-03-2022,23:55:26,0:09:32,-1425.900000
91,26-03-2022,23:57:58,0:10:05,-1427.883333
150,04-03-2022,23:49:49,0:02:33,-1427.266667
360,12-02-2022,23:56:42,0:04:26,-1432.266667
404,24-03-2022,23:46:00,0:03:47,-1422.216667
495,17-03-2022,23:49:29,0:02:38,-1426.850000
518,23-03-2022,23:55:47,0:05:22,-1430.416667
525,20-03-2022,23:55:32,0:09:43,-1425.816667
685,13-03-2022,23:47:32,0:00:12,-1427.333333


In [98]:
# Creatting a pickup rush hour feature

df_clean['is_peak_pickup'] = (df_clean['pickup_hour'].isin([11,12,13,14,19,20,21,22]))

In [99]:
# insepcting the temporal features

temporal_features = [
    "pickup_hour",
    "pickup_minute",
    "pickup_day_of_week",
    "pickup_day_name",
    "pickup_is_weekend",
    "pickup_time_period",
    "pickup_hour_sin",
    "pickup_hour_cos",
    "is_peak_pickup",
    "order_to_pickup_minutes"
]

df_clean[temporal_features].describe(include="all")

,pickup_hour,pickup_minute,pickup_day_of_week,pickup_day_name,pickup_is_weekend,pickup_time_period,pickup_hour_sin,pickup_hour_cos,is_peak_pickup,order_to_pickup_minutes
count,41953.000000,41953.000000,41953.000000,41953,41953,41953,41953.000000,41953.000000,41953,40353.000000
unique,NaN,NaN,NaN,7,2,5,NaN,NaN,2,NaN
top,NaN,NaN,NaN,Wednesday,False,night,NaN,NaN,True,NaN
freq,NaN,NaN,NaN,6552,30407,21828,NaN,NaN,21928,NaN
mean,17.146116,30.599576,2.994041,NaN,NaN,NaN,-0.474043,0.140561,NaN,-16.037530
std,5.321796,17.623396,1.970272,NaN,NaN,NaN,0.552829,0.670763,NaN,189.788237
min,0.000000,0.000000,0.000000,NaN,NaN,NaN,-1.000000,-1.000000,NaN,-1439.500000
25%,14.000000,15.000000,1.000000,NaN,NaN,NaN,-0.965926,-0.500000,NaN,5.516667
50%,19.000000,32.000000,3.000000,NaN,NaN,NaN,-0.707107,0.258819,NaN,9.283333
75%,21.000000,46.000000,5.000000,NaN,NaN,NaN,-0.258819,0.707107,NaN,13.133333


In [100]:
df_clean[temporal_features].head(10)

,pickup_hour,pickup_minute,pickup_day_of_week,pickup_day_name,pickup_is_weekend,pickup_time_period,pickup_hour_sin,pickup_hour_cos,is_peak_pickup,order_to_pickup_minutes
0,11,45,5,Saturday,True,afternoon,0.258819,-9.659258e-01,True,11.933333
1,19,51,4,Friday,False,night,-0.965926,2.588190e-01,True,6.200000
2,8,48,5,Saturday,True,morning,0.866025,-5.000000e-01,False,15.816667
3,18,12,1,Tuesday,False,evening,-1.000000,-1.836970e-16,False,8.900000
4,13,45,5,Saturday,True,afternoon,-0.258819,-9.659258e-01,True,11.333333
5,21,30,4,Friday,False,night,-0.707107,7.071068e-01,True,6.866667
6,19,34,4,Friday,False,night,-0.965926,2.588190e-01,True,14.950000
7,17,31,0,Monday,False,evening,-0.965926,-2.588190e-01,False,2.616667
8,21,6,6,Sunday,True,night,-0.707107,7.071068e-01,True,9.716667
9,22,10,5,Saturday,True,night,-0.500000,8.660254e-01,True,11.450000


In [101]:
# Geospatial feature engineering
def haversine_distance_km(
    lat1,
    lon1,
    lat2,
    lon2
):
    """
    Calculate great-circle distance between two
    geographic coordinates in kilometers.
    """

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    earth_radius_km = 6371.0

    return earth_radius_km * c

In [102]:
df_clean["distance_km"] = haversine_distance_km(
    df_clean["Restaurant_latitude"],
    df_clean["Restaurant_longitude"],
    df_clean["Delivery_location_latitude"],
    df_clean["Delivery_location_longitude"]
)

In [103]:
df_clean['distance_km'].describe()

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: distance_km, dtype: float64

In [104]:
df_clean['distance_km'].sort_values(ascending=False).head(20)

23883    20.969489
14443    20.969489
831      20.969489
23452    20.969045
15297    20.969045
15436    20.969045
10293    20.969028
18165    20.969028
27857    20.969028
10523    20.968994
29791    20.968994
36718    20.968994
34234    20.968994
36226    20.968993
12724    20.968993
7760     20.968993
20634    20.968993
4073     20.968989
32442    20.968899
2451     20.968899
Name: distance_km, dtype: float64

In [105]:
df_clean['distance_km'].describe()

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: distance_km, dtype: float64

In [106]:
print("Zero_distance_deliveries :",(df_clean['distance_km'] == 0).sum())
print("Distance > 50 km :",(df_clean['distance_km'] > 50).sum())

Zero_distance_deliveries : 0
Distance > 50 km : 0


In [107]:
# we will create distance Bands
df_clean['distance_bands'] = pd.cut(df_clean['distance_km'],bins = [-np.inf,1,3,5,10,np.inf],labels  = ['very_short','short',
                                                                                                        'medium','long','very_long'])

In [108]:
df_clean['distance_bands'].value_counts(dropna=False)

distance_bands
very_long     19343
long          11307
medium         6983
short          4320
very_short        0
Name: count, dtype: int64

In [109]:
# Inspect the distribution of negative order-to-pickup durations

negative_durations = (
    df_clean.loc[df_clean['order_to_pickup_minutes'] < 0,"order_to_pickup_minutes"].sort_values() )

print("number of negative durations :",len(negative_durations))

print("\nMost negative values :")
print(negative_durations.head(20))

print("\nLeast negative values :")
print(negative_durations.tail(20))

number of negative durations : 794

Most negative values :
21234   -1439.500000
40303   -1439.466667
4167    -1439.366667
1892    -1439.266667
10376   -1439.183333
3337    -1439.183333
26085   -1438.850000
7792    -1438.750000
26372   -1438.700000
23441   -1438.616667
35285   -1438.466667
39059   -1438.450000
10622   -1438.383333
40881   -1438.300000
20800   -1438.200000
22279   -1438.200000
5781    -1438.100000
27825   -1438.083333
37937   -1438.016667
33216   -1438.016667
Name: order_to_pickup_minutes, dtype: float64

Least negative values :
21980   -0.133333
38801   -0.116667
33107   -0.116667
11288   -0.116667
8517    -0.116667
23630   -0.100000
2918    -0.100000
11396   -0.083333
23581   -0.083333
5817    -0.083333
28054   -0.066667
39896   -0.066667
13797   -0.050000
38802   -0.050000
3803    -0.033333
1713    -0.033333
5399    -0.033333
28626   -0.016667
33438   -0.016667
26392   -0.016667
Name: order_to_pickup_minutes, dtype: float64


In [110]:
negative_durations.describe()

count     794.000000
mean    -1286.584992
std       427.856753
min     -1439.500000
25%     -1431.050000
50%     -1427.383333
75%     -1424.875000
max        -0.016667
Name: order_to_pickup_minutes, dtype: float64

In [111]:
df_clean["distance_km"].quantile(
    [0, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

0.00     1.465067
0.10     2.980290
0.25     4.657655
0.50     9.193021
0.75    13.680920
0.90    19.090006
0.95    19.917248
0.99    20.831168
1.00    20.969489
Name: distance_km, dtype: float64

#  Final Feature Definition

The objective of this section is to define the candidate feature set that
will be used for machine learning.

Identifier fields, raw timestamp representations, and the original target
column are excluded from the initial feature matrix.

Engineered temporal and geospatial features are retained as candidate
predictors.

The final feature set will be validated during model development using
validation performance and feature-ablation experiments.

In [112]:
candidate_features = [
    # Driver features
    "Delivery_person_Age",
    "Delivery_person_Ratings",

    # Geography
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "distance_km",
    "distance_bands",

    # Temporal
    "pickup_hour",
    "pickup_minute",
    "pickup_day_of_week",
    "pickup_is_weekend",
    "pickup_time_period",
    "pickup_hour_sin",
    "pickup_hour_cos",
    "is_peak_pickup",

    # Order / operational
    "Weatherconditions",
    "Road_traffic_density",
    "Vehicle_condition",
    "Type_of_order",
    "Type_of_vehicle",
    "multiple_deliveries",
    "Festival",
    "City"
]

print("Number of candidate features:", len(candidate_features))

for feature in candidate_features:
    print(feature)

Number of candidate features: 24
Delivery_person_Age
Delivery_person_Ratings
Restaurant_latitude
Restaurant_longitude
Delivery_location_latitude
Delivery_location_longitude
distance_km
distance_bands
pickup_hour
pickup_minute
pickup_day_of_week
pickup_is_weekend
pickup_time_period
pickup_hour_sin
pickup_hour_cos
is_peak_pickup
Weatherconditions
Road_traffic_density
Vehicle_condition
Type_of_order
Type_of_vehicle
multiple_deliveries
Festival
City


In [113]:
excluded_columns = [
    "ID",
    "Delivery_person_ID",
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
    "Time_taken(min)",
    "order_date",
    "order_time",
    "pickup_time",
    "pickup_timestamp",
    "order_timestamp",
    "target_delivery_time"
]

print("Excluded columns:")
for column in excluded_columns:
    print("-", column)

Excluded columns:
- ID
- Delivery_person_ID
- Order_Date
- Time_Orderd
- Time_Order_picked
- Time_taken(min)
- order_date
- order_time
- pickup_time
- pickup_timestamp
- order_timestamp
- target_delivery_time


In [114]:
# validating the feature List
missing_features = [feature for feature in candidate_features if feature not in df_clean.columns]
print("Missing candidate features :",missing_features)

Missing candidate features : []


In [115]:
X = df_clean[candidate_features].copy()
y = df_clean["target_delivery_time"].copy()

print('X shape :',X.shape)
print('y shape :',y.shape)



X shape : (41953, 24)
y shape : (41953,)


In [116]:
feature_type_report = pd.DataFrame({
    'feature': X.columns,
    'dtype ':X.dtypes.astype(str).values,
    'missing':X.isna().sum().values,
    'unique_values':X.nunique(dropna=False).values
})

feature_type_report

,feature,dtype,missing,unique_values
0,Delivery_person_Age,Int64,1719,23
1,Delivery_person_Ratings,Float64,1763,29
2,Restaurant_latitude,float64,0,388
3,Restaurant_longitude,float64,0,388
4,Delivery_location_latitude,float64,0,4362
5,Delivery_location_longitude,float64,0,4362
6,distance_km,float64,0,4362
7,distance_bands,category,0,4
8,pickup_hour,int32,0,17
9,pickup_minute,int32,0,60


In [117]:
# categorical Values

categorical_features = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City",
    "pickup_time_period",
    "distance_bands"
]



In [118]:
# Numerical Features
numerical_features = [
    feature
    for feature in candidate_features
    if feature not in categorical_features
]

In [119]:
print("Categorical_features :")
print(categorical_features)
print('\nNumerical features :')
print(numerical_features)

Categorical_features :
['Weatherconditions', 'Road_traffic_density', 'Type_of_order', 'Type_of_vehicle', 'Festival', 'City', 'pickup_time_period', 'distance_bands']

Numerical features :
['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'distance_km', 'pickup_hour', 'pickup_minute', 'pickup_day_of_week', 'pickup_is_weekend', 'pickup_hour_sin', 'pickup_hour_cos', 'is_peak_pickup', 'Vehicle_condition', 'multiple_deliveries']


In [120]:
# checking categorical Values

for column in categorical_features:
    print(f"\n{'=' * 60}")
    print(column)
    print(X[column].value_counts(dropna=False))


Weatherconditions
Weatherconditions
Fog           7012
Stormy        6974
Cloudy        6932
Sandstorms    6906
Windy         6832
Sunny         6728
NaN            569
Name: count, dtype: Int64

Road_traffic_density
Road_traffic_density
Low       14200
Jam       13043
Medium    10084
High       4071
NaN         555
Name: count, dtype: Int64

Type_of_order
Type_of_order
Snack     10616
Meal      10524
Drinks    10445
Buffet    10368
Name: count, dtype: Int64

Type_of_vehicle
Type_of_vehicle
motorcycle          24396
scooter             14029
electric_scooter     3468
bicycle                60
Name: count, dtype: Int64

Festival
Festival
No     40916
Yes      822
NaN      215
Name: count, dtype: Int64

City
City
Metropolitian    31411
Urban             9279
NaN               1114
Semi-Urban         149
Name: count, dtype: Int64

pickup_time_period
pickup_time_period
night         21828
evening        9327
morning        5200
afternoon      4386
late_night     1212
Name: count, dtype: i

In [121]:
X[numerical_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Delivery_person_Age,40234.0,29.56333,5.812361,15.0,25.0,30.0,35.0,50.0
Delivery_person_Ratings,40190.0,4.632386,0.335541,1.0,4.5,4.7,4.9,6.0
Restaurant_latitude,41953.0,18.911397,5.467829,9.957144,12.986047,19.065838,22.751234,30.914057
Restaurant_longitude,41953.0,76.923408,3.50291,72.768726,73.89852,76.618203,78.368855,88.433452
Delivery_location_latitude,41953.0,18.975064,5.469616,9.967144,13.065996,19.123249,22.82004,31.054057
Delivery_location_longitude,41953.0,76.987074,3.503073,72.778726,73.940327,76.663622,78.405467,88.563452
distance_km,41953.0,9.720284,5.603083,1.465067,4.657655,9.193021,13.68092,20.969489
pickup_hour,41953.0,17.146116,5.321796,0.0,14.0,19.0,21.0,23.0
pickup_minute,41953.0,30.599576,17.623396,0.0,15.0,32.0,46.0,59.0
pickup_day_of_week,41953.0,2.994041,1.970272,0.0,1.0,3.0,5.0,6.0


In [122]:
feature_policy = pd.DataFrame([
    ["Delivery_person_Age", "Driver", "Numeric", "Keep"],
    ["Delivery_person_Ratings", "Driver", "Numeric", "Keep"],
    ["Restaurant_latitude", "Geography", "Numeric", "Keep"],
    ["Restaurant_longitude", "Geography", "Numeric", "Keep"],
    ["Delivery_location_latitude", "Geography", "Numeric", "Keep"],
    ["Delivery_location_longitude", "Geography", "Numeric", "Keep"],
    ["distance_km", "Geography", "Numeric", "Keep"],
    ["distance_bands", "Geography", "Categorical", "Candidate"],
    ["pickup_hour", "Temporal", "Numeric", "Keep"],
    ["pickup_minute", "Temporal", "Numeric", "Candidate"],
    ["pickup_day_of_week", "Temporal", "Numeric", "Keep"],
    ["pickup_is_weekend", "Temporal", "Boolean", "Keep"],
    ["pickup_time_period", "Temporal", "Categorical", "Candidate"],
    ["pickup_hour_sin", "Temporal", "Numeric", "Candidate"],
    ["pickup_hour_cos", "Temporal", "Numeric", "Candidate"],
    ["is_peak_pickup", "Temporal", "Boolean", "Candidate"],
    ["Weatherconditions", "Environment", "Categorical", "Validate"],
    ["Road_traffic_density", "Operations", "Categorical", "Keep"],
    ["Vehicle_condition", "Vehicle", "Numeric", "Keep"],
    ["Type_of_order", "Order", "Categorical", "Keep"],
    ["Type_of_vehicle", "Vehicle", "Categorical", "Keep"],
    ["multiple_deliveries", "Operations", "Numeric", "Keep"],
    ["Festival", "Context", "Categorical", "Keep"],
    ["City", "Location", "Categorical", "Keep"],
], columns=[
    "feature",
    "feature_group",
    "data_type",
    "initial_policy"
])

feature_policy

,feature,feature_group,data_type,initial_policy
0,Delivery_person_Age,Driver,Numeric,Keep
1,Delivery_person_Ratings,Driver,Numeric,Keep
2,Restaurant_latitude,Geography,Numeric,Keep
3,Restaurant_longitude,Geography,Numeric,Keep
4,Delivery_location_latitude,Geography,Numeric,Keep
5,Delivery_location_longitude,Geography,Numeric,Keep
6,distance_km,Geography,Numeric,Keep
7,distance_bands,Geography,Categorical,Candidate
8,pickup_hour,Temporal,Numeric,Keep
9,pickup_minute,Temporal,Numeric,Candidate


In [123]:
df_clean.to_csv(PROCESSED_DATA_PATH / "cleaned_data.csv", index=False)
